In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.stats import wilcoxon, ranksums
from statsmodels.stats.multitest import multipletests
import plotly.express as px

from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

In [ ]:
project_path = projectpath = r'${PROJECT_ROOT}' 
model = 'test2' # test 1 :short term stress, test 2: long term stress
group = 'depressed'
mice_ids = ['list of miceIDs']
condition = 'pre'

# K
K_range = range(3, 8)  # K = 3, 4, 5, 6, 7

columns = [f'{dim}D_K{k}' for dim in [2, 3] for k in K_range]
KNN_accuracy_df = pd.DataFrame(index=mice_ids, columns=columns)

for miceID in mice_ids:
    VERBOSE = (miceID == mice_ids[0]) # EN: translated release comment.

    signal_path_TST_align = os.path.join(projectpath, 'signal_data', 'test2', condition, miceID, 'signal_save', f'{miceID}_{condition}_aligned_table_TST.csv')
    signal_path_OFT_align = os.path.join(projectpath, 'signal_data', 'test2', condition, miceID, 'signal_save', f'{miceID}_{condition}_aligned_table_OFT.csv')

    # Load aligned data
    tst_data_by_time = pd.read_csv(signal_path_TST_align).to_numpy()
    oft_data_by_time = pd.read_csv(signal_path_OFT_align).to_numpy()

    print(f'\nProcessing {miceID}: TST shape {tst_data_by_time.shape}, OFT shape {oft_data_by_time.shape}')

    # Constrain to the minimum time length and create labels
    time_constrain = min(tst_data_by_time.shape[0], oft_data_by_time.shape[0])
    tst_data_by_time = tst_data_by_time[:time_constrain, :]
    oft_data_by_time = oft_data_by_time[:time_constrain, :]
    y = np.hstack([np.zeros(tst_data_by_time.shape[0]), np.ones(oft_data_by_time.shape[0])])

    # Scale each dataset independently
    scaler_tst = MinMaxScaler()
    scaler_oft = MinMaxScaler()
    tst_scaled = scaler_tst.fit_transform(tst_data_by_time)
    plt.imshow(tst_scaled, aspect='auto', cmap='coolwarm', interpolation='none')
    plt.title(f'{miceID}: Scaled Neural Activity (TST)')
    plt.colorbar()
    plt.show()
    

    oft_scaled = scaler_oft.fit_transform(oft_data_by_time)
    plt.imshow(oft_scaled, aspect='auto', cmap='coolwarm', interpolation='none')
    plt.title(f'{miceID}: Scaled Neural Activity (OFT)')
    plt.colorbar()
    plt.show()
    
    X_scaled = np.vstack([tst_scaled, oft_scaled])

    if VERBOSE:
        plt.figure(figsize=(10, 4))
        plt.imshow(X_scaled, aspect='auto', cmap='coolwarm', interpolation='none')
        plt.title(f'{miceID}: Combined Scaled Neural Activity (TST + OFT)')
        plt.colorbar()
        plt.show()

    # PCA
    pca = PCA(n_components=3)
    X_pca = pca.fit_transform(X_scaled)
    print(f'Explained variance ratios: {pca.explained_variance_ratio_}')

    if VERBOSE:
        # Plotly 3D
        fig = px.scatter_3d(
            x=X_pca[:, 0], y=X_pca[:, 1], z=X_pca[:, 2],
            color=y, opacity=0.7,
            color_discrete_sequence=['blue', 'red'],
            title=f'{miceID}: PCA Space (TST vs OFT)'
        )
        fig.update_traces(marker=dict(size=2))
        fig.show()

        # Matplotlib 3D
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(X_pca[y==0, 0], X_pca[y==0, 1], X_pca[y==0, 2], c='blue', label='TST', alpha=0.6)
        ax.scatter(X_pca[y==1, 0], X_pca[y==1, 1], X_pca[y==1, 2], c='red', label='OFT', alpha=0.6)
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
        ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]:.1%})')
        ax.set_title(f'{miceID}: PCA Trajectories')
        ax.legend()
        plt.show()

    # --- KNN robustness test over K = 3 to 7 ---
    for k in K_range:
        # 2D: PC1 + PC2
        X_2d = X_pca[:, :2]
        X_train, X_test, y_train, y_test = train_test_split(
            X_2d, y, test_size=0.5, stratify=y, random_state=42 # EN: translated release comment.
        )
        knn_2d = KNeighborsClassifier(n_neighbors=k)
        knn_2d.fit(X_train, y_train)
        acc_2d = accuracy_score(y_test, knn_2d.predict(X_test))
        KNN_accuracy_df.loc[miceID, f'2D_K{k}'] = acc_2d

        # 3D: PC1 + PC2 + PC3
        X_3d = X_pca[:, :3]
        X_train, X_test, y_train, y_test = train_test_split(
            X_3d, y, test_size=0.5, stratify=y, random_state=42
        )
        knn_3d = KNeighborsClassifier(n_neighbors=k)
        knn_3d.fit(X_train, y_train)
        acc_3d = accuracy_score(y_test, knn_3d.predict(X_test))
        KNN_accuracy_df.loc[miceID, f'3D_K{k}'] = acc_3d

        if VERBOSE and k == 5: # EN: translated release comment.
            # 2D plot
            plt.figure(figsize=(6, 5))
            plt.scatter(X_test[y_test==0, 0], X_test[y_test==0, 1], c='blue', label='TST', alpha=0.6)
            plt.scatter(X_test[y_test==1, 0], X_test[y_test==1, 1], c='red', label='OFT', alpha=0.6)
            plt.title(f'{miceID}: 2D KNN (K={k}), Acc={acc_2d:.2%}')
            plt.legend()
            plt.show()

            # 3D plot
            fig = plt.figure(figsize=(7, 6))
            ax = fig.add_subplot(111, projection='3d')
            ax.scatter(X_test[y_test==0, 0], X_test[y_test==0, 1], X_test[y_test==0, 2], c='blue', label='TST', alpha=0.6)
            ax.scatter(X_test[y_test==1, 0], X_test[y_test==1, 1], X_test[y_test==1, 2], c='red', label='OFT', alpha=0.6)
            ax.set_title(f'{miceID}: 3D KNN (K={k}), Acc={acc_3d:.2%}')
            ax.legend()
            plt.show()

    mean_2d = KNN_accuracy_df.loc[miceID, [f'2D_K{k}' for k in K_range]].mean()
    mean_3d = KNN_accuracy_df.loc[miceID, [f'3D_K{k}' for k in K_range]].mean()
    print(f"{miceID} → Avg 2D Acc: {mean_2d:.2%}, Avg 3D Acc: {mean_3d:.2%}")

In [ ]:
save_path = os.path.join(projectpath, 'signal_data', 'test2',condition,'group',group)
KNN_accuracy_df.to_csv(os.path.join(save_path, f'{group}_{condition}_KNN_accuracy_summary.csv'))
print(f'KNN accuracy summary saved to {os.path.join(save_path, f"{group}_{condition}_KNN_accuracy_summary.csv")}')